In [0]:
VECTOR_DB_PATH = "/Volumes/workspace/legal_data/vector_db/"

In [0]:
%pip install sentence-transformers chromadb

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

In [0]:
GOLD_PATH = "/Volumes/workspace/legal_data/gold/legal_chunks/"

gold_df = spark.read.format("delta").load(GOLD_PATH)

gold_df = gold_df.select(
    "chunk_id",
    "chunk_text",
    "act_name",
    "section_number",
    "category",
    "file_name"
)

gold_df.display()

chunk_id,chunk_text,act_name,section_number,category,file_name
780a036f-70c6-41ed-b6c5-5cec19f6ab92,"fence or the Appellate Court, as the case may be, shall require the accused to appear before execute a bond or bail bond, to appear before the higher Court as and when such Court next Appellate Court. issues notice in respect of any appeal or petition filed against the judgment of the respective Court and such bond shall be in force for six months. (2)If such accused fails to appear, the bond stand forfeited and the procedure under section 491 shall apply. Direction for 482. (1) When any person has reason to believe that he may be arrested on an grant of bail accusation of having committed a non-bailable offence, he may apply to the High Court or to person apprehending the Court of Session for a direction under this section; and that Court may, if it thinks fit, arrest. direct that in the event of such arrest, he shall be released on bail. (2) When the High Court or the Court of Session makes a direction under sub-section (1), it may include such conditions in such directions in the light of the facts of the particular case, as it may think fit, including— (i) a condition that the person shall make himself available for interrogation by a police officer as and when required; (ii) a condition that the person shall not, directly or indirectly, make any inducement, threat or promise to any person acquainted with the facts of the case so as to dissuade him from disclosing such facts to the Court or to any police officer; (iii) a condition that the person shall not leave India without the previous permission of the Court; (iv) such other condition as may be imposed under sub-section (3) of section 480, as if the bail were granted under that section.",Bharatiya Nagarik Suraksha Sanhita 2023,Chapter VII,criminal_law,Bharatiya Nagarik Suraksha Sanhita_2023.pdf
d15ffd5f-66d4-4a19-ac41-bfa4f90a84e1,"on shall not leave India without the previous permission of the Court; (iv) such other condition as may be imposed under sub-section (3) of section 480, as if the bail were granted under that section. (3)If such person is thereafter arrested without warrant by an officer in charge of a police station on such accusation, and is prepared either at the time of arrest or at any time while in the custody of such officer to give bail, he shall be released on bail; and if a Magistrate taking cognizance of such offence decides that a warrant should be issued in the first instance against that person, he shall issue a bailable warrant in conformity with the direction of the Court under sub-section (1). (4)Nothing in this section shall apply to any case involving the arrest of any person on accusation of having committed an offence under section 65 and sub-section (2) of section 70 of the Bharatiya Nyaya Sanhita, 2023. Special powers 483.(1) A High Court or Court of Session may direct,— of High Court or Court of (a)that any person accused of an offence and in custody be released on bail, Session and if the offence is of the nature specified in sub-section (3) of section 480, may regarding bail. impose any condition which it considers necessary for the purposes mentioned in that sub-section;",Bharatiya Nagarik Suraksha Sanhita 2023,Chapter VII,criminal_law,Bharatiya Nagarik Suraksha Sanhita_2023.pdf
b53d213e-4cf7-410b-be0a-e7eda7eb0b0b,"; (k) takes cognizance of an offence under clause (c) of sub-section (1) of section 210; (l)tries an offender; (m) tries an offender summarily; (n)passes a sentence, under section 364, on proceedings recorded by another Magistrate; (o)decides an appeal; (p)calls, under section 438, for proceedings; or (q)revises an order passed under section 491, his proceedings shall be void. Proceedings in 508. No finding, sentence or order of any Criminal Court shall be set aside merely on wrong place. the ground that the inquiry, trial or other proceedings in the course of which it was arrived at or passed, took place in a wrong sess

In [0]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [0]:
VECTOR_DB_PATH = "/local_disk0/tmp/chroma_db"

/Volumes/workspace/legal_data/vector_db/

In [0]:
import os

os.makedirs(VECTOR_DB_PATH, exist_ok=True)

In [0]:
import chromadb

client = chromadb.PersistentClient(path=VECTOR_DB_PATH)

collection = client.get_or_create_collection(
    name="legal_knowledge",
    metadata={"hnsw:space": "cosine"}
)

print("Vector DB ready")

Vector DB ready


In [0]:
def safe_str(value):
    if value is None:
        return ""
    return str(value)

In [0]:
rows = gold_df.toLocalIterator()

In [0]:
batch_size = 100
batch = []

for row in rows:
    batch.append(row)

    if len(batch) == batch_size:
        texts = [safe_str(r.chunk_text) for r in batch]
        ids = [safe_str(r.chunk_id) for r in batch]

        embeddings = model.encode(texts).tolist()

        metadata = []
        for r in batch:
            metadata.append({
                "act_name": safe_str(r.act_name),
                "section": safe_str(r.section_number),
                "category": safe_str(r.category),
                "source": safe_str(r.file_name),
            })

        collection.add(
            ids=ids,
            documents=texts,
            embeddings=embeddings,
            metadatas=metadata
        )

        print(f"Inserted {len(batch)} records")
        batch = []

# insert remaining rows
if batch:
    texts = [safe_str(r.chunk_text) for r in batch]
    ids = [safe_str(r.chunk_id) for r in batch]
    embeddings = model.encode(texts).tolist()

    metadata = []
    for r in batch:
        metadata.append({
            "act_name": safe_str(r.act_name),
            "section": safe_str(r.section_number),
            "category": safe_str(r.category),
            "source": safe_str(r.file_name),
        })

    collection.add(
        ids=ids,
        documents=texts,
        embeddings=embeddings,
        metadatas=metadata
    )

    print(f"Inserted final {len(batch)} records")

Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 records
Inserted 100 

In [0]:
collection.count()

5194

In [0]:
query = "What is the penalty under Motor Vehicles Act for not wearing helmet under section 129?"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=10
)

docs = results["documents"][0]

filtered_docs = [doc for doc in docs if "helmet" in doc.lower()]

print(filtered_docs[:3])

['-section (1) or sub-section (2) by a police officer, the owner of the vehicle shall be responsible for all towing costs, besides any other penalty. 128. Safety measures for drivers and pillion riders.—(1) No driver of a two-wheeled motor cycle shall carry more than one person in addition to himself on the motor cycle and no such person shall be carried otherwise than sitting on a proper seat securely fixed to the motor cycle behind the driver’s seat with appropriate safety measures. (2) In addition to the safety measures mentioned in sub-section (1), the Central Government may, prescribe other safety measures for the drivers of two-wheeled motor cycles and pillion riders thereon. 129. Wearing of protective headgear.—Every person driving or riding (otherwise than in a side car, on a motor cycle of any class or description) shall, while in a public place, wear 2[protective headgear conforming to the standards of Bureau of Indian Standards]: Provided that the provisions of this section 